In [13]:
import os
import pandas as pd
import numpy as np

# Load Cleaned Dataset
data_path = os.path.join('data', 'all_months_clean.csv')

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f" Cleaned dataset loaded successfully. Shape: {df.shape}")
else:
    raise FileNotFoundError(f"File not found at '{data_path}'. Please run step 02 first.")

# Ensure chronological ordering by product and time index
df = df.sort_values(['Product_Name', 'month_idx']).reset_index(drop=True)

 Cleaned dataset loaded successfully. Shape: (798, 55)


In [14]:
# Price Lag & Volatility Features
price_targets = ['Avg_Price', 'Min_Price', 'Max_Price']

for col in price_targets:
    # 1-month and 2-month price lags per product
    df[f'{col}_lag1'] = df.groupby('Product_Name')[col].shift(1)
    df[f'{col}_lag2'] = df.groupby('Product_Name')[col].shift(2)
    
    # Rolling 3-month mean and std dev
    df[f'{col}_roll3_mean'] = df.groupby('Product_Name')[col].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    df[f'{col}_roll3_std'] = df.groupby('Product_Name')[col].transform(lambda x: x.shift(1).rolling(3, min_periods=1).std())

# Price spread ratio (Max - Min spread relative to Avg Price)
df['price_spread'] = df['Max_Price'] - df['Min_Price']
df['price_spread_ratio'] = np.where(df['Avg_Price'] > 0, df['price_spread'] / df['Avg_Price'], np.nan)

In [15]:
# Supply & Volume Lag Features
# Use TOTAL_sources / Volume as supply measures
df['Volume_lag1'] = df.groupby('Product_Name')['Volume'].shift(1)
df['TOTAL_sources_lag1'] = df.groupby('Product_Name')['TOTAL_sources'].shift(1)

# Supply change ratio relative to previous month
df['volume_change_ratio'] = np.where(df['Volume_lag1'] > 0, (df['Volume'] - df['Volume_lag1']) / df['Volume_lag1'], np.nan)

In [16]:
# Cyclical Month Encoding (BS Calendar Seasonality)
# Map month_idx (1-10) cyclically using Sine and Cosine
df['sin_month'] = np.sin(2 * np.pi * df['month_idx'] / 12)
df['cos_month'] = np.cos(2 * np.pi * df['month_idx'] / 12)

In [17]:
# Supply Origin Share Features
# Define key aggregated origin pillars
import_cols = [c for c in ['India', 'China', 'Bhutan'] if c in df.columns]
if import_cols:
    df['total_imports'] = df[import_cols].fillna(0).sum(axis=1)
    df['import_ratio'] = np.where(df['TOTAL_sources'] > 0, df['total_imports'] / df['TOTAL_sources'], 0)

# Local vs Kathmandu supply ratio (if available in schema)
if 'Kathmandu' in df.columns:
    df['ktm_supply_share'] = np.where(df['TOTAL_sources'] > 0, df['Kathmandu'].fillna(0) / df['TOTAL_sources'], 0)

In [18]:
# Export Feature-Engineered Dataset
output_path = os.path.join('data', 'all_months_features.csv')
df.to_csv(output_path, index=False)

print(f" Feature engineering complete!")
print(f"Saved dataset: '{output_path}' | Shape: {df.shape[0]} rows, {df.shape[1]} columns")

 Feature engineering complete!
Saved dataset: 'data/all_months_features.csv' | Shape: 798 rows, 77 columns
